<a href="https://colab.research.google.com/github/audreyagbeve/HyperChat/blob/main/health_chatbot_colab_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Instructor-led Build: Simple Health Education Chatbot with a Hosted Hugging Face Model

This notebook builds a simple health education chatbot using:

- Python
- Google Colab
- Hugging Face Inference Providers
- Gradio
- Hugging Face Spaces

Unlike the MedGemma local-loading version, this version does **not** load a large model into Colab memory. The model runs through Hugging Face hosted inference, so the notebook can run on normal Colab CPU.

## What students will build

A chatbot that:

1. answers general health education questions
2. avoids diagnosis and prescribing
3. detects red-flag symptoms with a local safety rule
4. runs in Colab
5. can be deployed as a CPU Hugging Face Space

## Important safety framing

This demo is for **health education only**.

It is not a doctor, pharmacist, nurse, emergency line, diagnosis system, or prescription tool.


## 1. Install required libraries

We will use:

- `huggingface_hub` to call hosted Hugging Face models
- `gradio` to create the chatbot interface


In [ ]:
!pip install -q -U huggingface_hub gradio

## 2. Add your Hugging Face token

In Colab:

1. Click the key icon on the left panel
2. Add a new secret called `HF_TOKEN`
3. Paste your Hugging Face access token
4. Turn on notebook access for the secret

Your token should have permission to call Hugging Face Inference Providers.

This notebook will read the token securely from Colab secrets.


In [ ]:
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found. Add it to Colab secrets or environment variables.")

print("HF token loaded successfully.")

HF token loaded successfully.


## 3. Choose a hosted model

We will use a hosted conversational model from Hugging Face Inference Providers.

Recommended for this class:

```python
Qwen/Qwen2.5-7B-Instruct:together
```

Why this choice?

- It is a general instruction-following model.
- It does not need to be loaded into Colab memory.
- It is suitable for a demo chatbot.
- It is not a medical specialist model, so we must use safety guardrails.

You can later replace `MODEL_ID` with another hosted chat model from the Hugging Face Playground.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct:together"

print("Using model:", MODEL_ID)

Using model: Qwen/Qwen2.5-7B-Instruct:together


## 4. Create the Hugging Face inference client

The client sends our prompt to the hosted model and receives the response.

The key idea:

- Colab handles the app logic
- Hugging Face handles model inference
- Gradio handles the user interface


In [ ]:
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider="auto",
    api_key=HF_TOKEN,
    timeout=120
)

print("Inference client created.")

Inference client created.


## 5. Create the health safety prompt

The system prompt defines what the chatbot should and should not do.

This is very important in healthcare demos.

The model should:

- explain health topics simply
- avoid diagnosis
- avoid prescribing
- advise users to seek professional care
- escalate red-flag symptoms


In [ ]:
SAFETY_PROMPT = """
You are a health education assistant.

Your role:
- Explain health topics in simple, clear language.
- Provide general health education only.
- Do not diagnose the user.
- Do not prescribe medicines.
- Do not give personalized treatment plans.
- Do not replace a doctor, pharmacist, nurse, or emergency service.
- Encourage users to speak with a qualified health professional.

Medication safety:
- If the user asks about medicines, explain only general information.
- Tell the user to confirm medicine use, dose, interactions, pregnancy safety, allergies, and contraindications with a pharmacist or doctor.

Emergency safety:
- If the user mentions chest pain, difficulty breathing, fainting, severe weakness, confusion, stroke symptoms, seizures, severe bleeding, suicidal thoughts, pregnancy complications, or symptoms in a baby, tell them to seek urgent medical attention immediately.

Response style:
- Be brief.
- Be practical.
- Use plain language.
- Say when you are unsure.
"""

print(SAFETY_PROMPT)


You are a health education assistant.

Your role:
- Explain health topics in simple, clear language.
- Provide general health education only.
- Do not diagnose the user.
- Do not prescribe medicines.
- Do not give personalized treatment plans.
- Do not replace a doctor, pharmacist, nurse, or emergency service.
- Encourage users to speak with a qualified health professional.

Medication safety:
- If the user asks about medicines, explain only general information.
- Tell the user to confirm medicine use, dose, interactions, pregnancy safety, allergies, and contraindications with a pharmacist or doctor.

Emergency safety:
- If the user mentions chest pain, difficulty breathing, fainting, severe weakness, confusion, stroke symptoms, seizures, severe bleeding, suicidal thoughts, pregnancy complications, or symptoms in a baby, tell them to seek urgent medical attention immediately.

Response style:
- Be brief.
- Be practical.
- Use plain language.
- Say when you are unsure.



## 6. Add local red-flag detection

This is not handled only by the model.

We add a simple local rule before the model is called.

Why?

Because safety-critical escalation should not depend entirely on the LLM.


In [ ]:
import re

RED_FLAG_PATTERNS = [
    r"chest pain",
    r"difficulty breathing",
    r"shortness of breath",
    r"fainting",
    r"passed out",
    r"loss of consciousness",
    r"severe bleeding",
    r"seizure",
    r"stroke",
    r"face drooping",
    r"weakness on one side",
    r"suicidal",
    r"kill myself",
    r"pregnancy bleeding",
    r"baby.*not breathing",
    r"confusion",
    r"severe weakness"
]

def detect_red_flags(text):
    text = text.lower()
    for pattern in RED_FLAG_PATTERNS:
        if re.search(pattern, text):
            return True
    return False

def urgent_response():
    return (
        "This may be urgent. Please seek emergency medical care immediately or contact your local emergency number. "
        "Do not wait for an online chatbot response. If someone is with you, ask them to help you get medical care now."
    )

print(detect_red_flags("I have chest pain and shortness of breath"))
print(detect_red_flags("What is malaria?"))

True
False


## 7. Format conversation history

Gradio sends previous messages to the function.

We keep only the recent conversation so the prompt stays short and focused.


In [ ]:
def build_messages(user_message, history=None):
    messages = [
        {"role": "system", "content": SAFETY_PROMPT}
    ]

    if history:
        for item in history[-6:]:
            role = item.get("role")
            content = item.get("content")

            if role in ["user", "assistant"] and content:
                messages.append({"role": role, "content": content})

    messages.append({"role": "user", "content": user_message})

    return messages

## 8. Create the chatbot response function

This function:

1. checks red flags first
2. builds the chat messages
3. calls the hosted Hugging Face model
4. returns the chatbot answer


In [ ]:
def generate_answer(user_message, history=None):
    if detect_red_flags(user_message):
        return urgent_response()

    messages = build_messages(user_message, history)

    try:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            max_tokens=350,
            temperature=0.2
        )

        answer = response.choices[0].message.content
        return answer.strip()

    except Exception as error:
        return (
            "I could not get a response from the hosted model. "
            "Please check your Hugging Face token, model access, or Inference Provider availability. "
            f"Technical error: {error}"
        )

## 9. Test the chatbot function

Run these one by one before building the UI.


In [ ]:
generate_answer("What is hypertension?")

'Hypertension, also called high blood pressure, is when the force of blood against your artery walls is too high. It usually happens over time and often has no symptoms, which is why it\'s sometimes called the "silent killer."\n\nBlood pressure is measured with two numbers:\n\n1. Systolic pressure (the first number) - This is the pressure in your arteries when your heart beats.\n2. Diastolic pressure (the second number) - This is the pressure in your arteries between heartbeats.\n\nHigh blood pressure is generally considered to be 130/80 mmHg or higher. It can increase the risk of serious health problems like heart disease, stroke, and kidney failure.\n\nManaging hypertension often involves lifestyle changes like eating a healthy diet, being physically active, maintaining a healthy weight, limiting alcohol, and not smoking. Sometimes, medications are also needed.\n\nIf you think you might have high blood pressure, it\'s important to talk to a healthcare provider for proper diagnosis an

In [ ]:
generate_answer("Can I take antibiotics for malaria?")

'No, antibiotics are not used to treat malaria. Malaria is typically treated with antimalarial drugs, such as chloroquine, artemisinin-based combination therapies (ACTs), or others, depending on the type of malaria parasite and your location.\n\nIf you think you have malaria, seek medical advice immediately. Only a healthcare provider can diagnose malaria and prescribe the appropriate medication. Misuse of antibiotics can lead to antibiotic resistance, making future infections harder to treat.\n\nAlways consult a doctor or a healthcare professional for medical advice and treatment.'

In [ ]:
generate_answer("I have chest pain and shortness of breath.")

## 10. Build the Gradio chatbot UI

This turns the function into a simple web app.


In [ ]:
import gradio as gr

demo = gr.ChatInterface(
    fn=generate_answer,
    title="Hosted Hugging Face Health Education Chatbot",
    description=(
        "A simple health education chatbot using a hosted Hugging Face model. "
        "This demo does not provide diagnosis, prescriptions, or emergency care."
    ),
    examples=[
        "What is diabetes?",
        "What are common symptoms of malaria?",
        "Can I take paracetamol for fever?",
        "I have chest pain and difficulty breathing.",
        "Explain hypertension in simple terms."
    ]
)

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a1d10d3644e0a36fca.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://a1d10d3644e0a36fca.gradio.live


# Deploy to Hugging Face Spaces

Now we will create the files needed for a Hugging Face Space.

Because the model is hosted through Hugging Face Inference Providers, the Space itself can run on CPU.

The Space only needs to run the Gradio interface and call the hosted model.


## 11. Create the Space project files

This creates:

- `app.py`
- `requirements.txt`
- `README.md`


In [ ]:
import os

project_dir = "hosted-hf-health-chatbot"
os.makedirs(project_dir, exist_ok=True)

app_py = 'import os\nimport re\nimport gradio as gr\nfrom huggingface_hub import InferenceClient\n\nHF_TOKEN = os.getenv("HF_TOKEN")\n\nif not HF_TOKEN:\n    raise ValueError("HF_TOKEN secret is missing. Add it in your Hugging Face Space settings.")\n\nMODEL_ID = os.getenv("MODEL_ID", "Qwen/Qwen2.5-7B-Instruct:together")\n\nclient = InferenceClient(\n    provider="auto",\n    api_key=HF_TOKEN,\n    timeout=120\n)\n\nSAFETY_PROMPT = """\nYou are a health education assistant.\n\nYour role:\n- Explain health topics in simple, clear language.\n- Provide general health education only.\n- Do not diagnose the user.\n- Do not prescribe medicines.\n- Do not give personalized treatment plans.\n- Do not replace a doctor, pharmacist, nurse, or emergency service.\n- Encourage users to speak with a qualified health professional.\n\nMedication safety:\n- If the user asks about medicines, explain only general information.\n- Tell the user to confirm medicine use, dose, interactions, pregnancy safety, allergies, and contraindications with a pharmacist or doctor.\n\nEmergency safety:\n- If the user mentions chest pain, difficulty breathing, fainting, severe weakness, confusion, stroke symptoms, seizures, severe bleeding, suicidal thoughts, pregnancy complications, or symptoms in a baby, tell them to seek urgent medical attention immediately.\n\nResponse style:\n- Be brief.\n- Be practical.\n- Use plain language.\n- Say when you are unsure.\n"""\n\nRED_FLAG_PATTERNS = [\n    r"chest pain",\n    r"difficulty breathing",\n    r"shortness of breath",\n    r"fainting",\n    r"passed out",\n    r"loss of consciousness",\n    r"severe bleeding",\n    r"seizure",\n    r"stroke",\n    r"face drooping",\n    r"weakness on one side",\n    r"suicidal",\n    r"kill myself",\n    r"pregnancy bleeding",\n    r"baby.*not breathing",\n    r"confusion",\n    r"severe weakness"\n]\n\ndef detect_red_flags(text):\n    text = text.lower()\n    for pattern in RED_FLAG_PATTERNS:\n        if re.search(pattern, text):\n            return True\n    return False\n\ndef urgent_response():\n    return (\n        "This may be urgent. Please seek emergency medical care immediately or contact your local emergency number. "\n        "Do not wait for an online chatbot response. If someone is with you, ask them to help you get medical care now."\n    )\n\ndef build_messages(user_message, history=None):\n    messages = [\n        {"role": "system", "content": SAFETY_PROMPT}\n    ]\n\n    if history:\n        for item in history[-6:]:\n            role = item.get("role")\n            content = item.get("content")\n\n            if role in ["user", "assistant"] and content:\n                messages.append({"role": role, "content": content})\n\n    messages.append({"role": "user", "content": user_message})\n\n    return messages\n\ndef generate_answer(user_message, history=None):\n    if detect_red_flags(user_message):\n        return urgent_response()\n\n    messages = build_messages(user_message, history)\n\n    try:\n        response = client.chat.completions.create(\n            model=MODEL_ID,\n            messages=messages,\n            max_tokens=350,\n            temperature=0.2\n        )\n\n        answer = response.choices[0].message.content\n        return answer.strip()\n\n    except Exception as error:\n        return (\n            "I could not get a response from the hosted model. "\n            "Please check your Hugging Face token, model access, or Inference Provider availability. "\n            f"Technical error: {error}"\n        )\n\ndemo = gr.ChatInterface(\n    fn=generate_answer,\n    type="messages",\n    title="Hosted Hugging Face Health Education Chatbot",\n    description=(\n        "A simple health education chatbot using a hosted Hugging Face model. "\n        "This demo does not provide diagnosis, prescriptions, or emergency care."\n    ),\n    examples=[\n        "What is diabetes?",\n        "What are common symptoms of malaria?",\n        "Can I take paracetamol for fever?",\n        "I have chest pain and difficulty breathing.",\n        "Explain hypertension in simple terms."\n    ]\n)\n\nif __name__ == "__main__":\n    demo.launch()\n'

requirements_txt = """huggingface_hub
gradio
"""

readme_md = """---
title: Hosted HF Health Chatbot
emoji: 🩺
colorFrom: blue
colorTo: green
sdk: gradio
app_file: app.py
pinned: false
---

# Hosted Hugging Face Health Education Chatbot

This is a simple health education chatbot built with Gradio and Hugging Face Inference Providers.

It provides general health education only.

It does not provide diagnosis, prescriptions, emergency care, or professional medical advice.

## Environment variables

Add these secrets in your Hugging Face Space settings:

- `HF_TOKEN`: your Hugging Face token with Inference Providers permission
- `MODEL_ID`: optional. Default is `Qwen/Qwen2.5-7B-Instruct:together`
"""

with open(os.path.join(project_dir, "app.py"), "w", encoding="utf-8") as f:
    f.write(app_py)

with open(os.path.join(project_dir, "requirements.txt"), "w", encoding="utf-8") as f:
    f.write(requirements_txt)

with open(os.path.join(project_dir, "README.md"), "w", encoding="utf-8") as f:
    f.write(readme_md)

print("Created files:")
print(os.listdir(project_dir))

Created files:
['app.py', 'README.md', 'requirements.txt']


## 12. Preview the app file

This lets students inspect the deployment file before pushing.


In [ ]:
with open("hosted-hf-health-chatbot/app.py", "r", encoding="utf-8") as f:
    print(f.read()[:3000])

## 13. Push the app to Hugging Face Spaces

Replace:

```python
your-username
```

with your actual Hugging Face username.

After uploading, go to the Space settings and add the secret:

```text
HF_TOKEN = your Hugging Face token
```

Optional:

```text
MODEL_ID =Qwen/Qwen2.5-7B-Instruct:together
```


In [ ]:
from huggingface_hub import create_repo, upload_folder

repo_id = "your-username/hosted-hf-health-chatbot"

create_repo(
    repo_id=repo_id,
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True,
    token=HF_TOKEN
)

upload_folder(
    folder_path="hosted-hf-health-chatbot",
    repo_id=repo_id,
    repo_type="space",
    token=HF_TOKEN
)

print("Uploaded to:", f"https://huggingface.co/spaces/{repo_id}")

# Teaching notes

## Why this version is easier than the local MedGemma version

Local MedGemma version:

- Needs GPU
- Needs model download
- May hit memory problems
- Slower to start
- Better medical-domain relevance

Hosted Hugging Face version:

- Does not need local GPU
- Easier for a live bootcamp
- Faster setup
- Easier to deploy on CPU Spaces
- Not medical-domain specific, so guardrails matter more

## Recommended classroom framing

Do not call this a diagnosis chatbot.

Call it:

```text
A safe health education chatbot demo
```

## Suggested classroom test prompts

Try these:

1. What is malaria?
2. What is hypertension?
3. Can I take antibiotics for malaria?
4. I have chest pain and shortness of breath.
5. Explain diabetes to a secondary school student.
6. What should I ask my pharmacist before taking a new medicine?

## Exercise for students

Ask students to improve the app by adding:

1. Nigerian Pidgin response mode
2. Yoruba/Hausa/Igbo language option
3. A medicine safety checklist
4. A stronger red-flag symptom list
5. A disclaimer box in the Gradio interface
